In [3]:
import pandas as pd

df = pd.read_excel("./../csv/library_id_cond.xlsx")

In [4]:
df['library_id'] = df['library_id'].str.replace("_","-")

In [5]:
df = df[df['cond']!= "UNC"]

In [6]:
df

,library_id,cond
0,18-57617-A1,IPF
1,20-33940-B2,IPF
2,20-24241-A2,IPF
3,19-35057-C3,NSIP
4,20-17688-B2,NSIP
5,20-28197-A1,IPF
6,20-22642-A1,NSIP
7,20-41501-C1,IPF
8,20-33362-C4,NSIP
10,20-41615-B1,IPF


In [7]:
df['IPF_onehot']  = (df['cond']=='IPF').astype(int)

In [8]:
df

,library_id,cond,IPF_onehot
0,18-57617-A1,IPF,1
1,20-33940-B2,IPF,1
2,20-24241-A2,IPF,1
3,19-35057-C3,NSIP,0
4,20-17688-B2,NSIP,0
5,20-28197-A1,IPF,1
6,20-22642-A1,NSIP,0
7,20-41501-C1,IPF,1
8,20-33362-C4,NSIP,0
10,20-41615-B1,IPF,1


In [9]:
df = df.reset_index(drop = True)

In [10]:
df.shape

(29, 3)

In [11]:
from sklearn.model_selection import StratifiedKFold  # or KFold if labels are not categorical

k = 6  # number of folds
y = df["IPF_onehot"]  # or your label column

skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(df, y)):
    colname = f"fold_{fold_idx}"
    
    df.loc[train_idx,colname] = "train"
    
    df.loc[test_idx, colname] = "test"


In [12]:
df

,library_id,cond,IPF_onehot,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5
0,18-57617-A1,IPF,1,train,train,train,train,test,train
1,20-33940-B2,IPF,1,test,train,train,train,train,train
2,20-24241-A2,IPF,1,train,train,test,train,train,train
3,19-35057-C3,NSIP,0,train,train,test,train,train,train
4,20-17688-B2,NSIP,0,train,train,train,train,train,test
5,20-28197-A1,IPF,1,test,train,train,train,train,train
6,20-22642-A1,NSIP,0,test,train,train,train,train,train
7,20-41501-C1,IPF,1,train,train,train,test,train,train
8,20-33362-C4,NSIP,0,train,train,test,train,train,train
9,20-41615-B1,IPF,1,train,test,train,train,train,train


In [13]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import h5py
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

from trident.slide_encoder_models import ABMILSlideEncoder

# Set deterministic behavior
SEED = 1234
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class BinaryClassificationModel(nn.Module):
    def __init__(self, input_feature_dim=1536, n_heads=1, head_dim=512, dropout=0., gated=True, hidden_dim=256):
        super().__init__()
        self.feature_encoder = ABMILSlideEncoder(
            freeze=False,
            input_feature_dim=input_feature_dim, 
            n_heads=n_heads, 
            head_dim=head_dim, 
            dropout=dropout, 
            gated=gated
        )
        self.classifier = nn.Sequential(
            nn.Linear(input_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, return_raw_attention=False):
        if return_raw_attention:
            features, attn = self.feature_encoder(x, return_raw_attention=True)
        else:
            features = self.feature_encoder(x)
        logits = self.classifier(features).squeeze(1)
        
        if return_raw_attention:
            return logits, attn
        
        return logits

# Initialize model
device = 'mps'
model = BinaryClassificationModel().to(device)

# Custom dataset
class H5Dataset(Dataset):
    def __init__(self, feats_path, df, split, test_idx = None, num_features=1536):
        #self.df = df[df["fold_0"] == split]
        #self.df = df.drop(test_idx)
        self.df = df[df[f"fold_{test_idx}"] == split]
        self.feats_path = feats_path
        self.num_features = num_features
        self.split = split
    
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with h5py.File(os.path.join(self.feats_path, row['library_id'] + '.h5'), "r") as f:
            features = torch.from_numpy(f["features"][:])

        if self.split == 'train':
            num_available = features.shape[0]
            if num_available >= self.num_features:
                indices = torch.randperm(num_available, generator=torch.Generator().manual_seed(SEED))[:self.num_features]
            else:
                indices = torch.randint(num_available, (self.num_features,), generator=torch.Generator().manual_seed(SEED))  # Oversampling
            features = features[indices]

        label = torch.tensor(row["IPF_onehot"], dtype=torch.float32)
        return features, label

# # Create dataloaders
# feats_path = '/Users/jk/Documents/Lab2/visium/python_analysis/trident/svs/trident_processed/20x_256px_0px_overlap/features_uni_v2'  #'./tutorial-3/cptac_ccrcc/20x_512px_0px_overlap/features_conch_v15'
# batch_size = 22
# train_loader = DataLoader(H5Dataset(feats_path, df, "train"), batch_size=batch_size, shuffle=False, worker_init_fn=lambda _: np.random.seed(SEED))
# test_loader = DataLoader(H5Dataset(feats_path, df, "test"), batch_size=1, shuffle=False, worker_init_fn=lambda _: np.random.seed(SEED))





/Users/jk/miniconda3/envs/trident/lib/python3.10/site-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


In [14]:
# Stuff that doesnt change 
feats_path = '/Users/jk/Documents/Lab2/visium/python_analysis/cpath/trident/svs/trident_processed/20x_256px_0px_overlap/features_uni_v2'  #'./tutorial-3/cptac_ccrcc/20x_512px_0px_overlap/features_conch_v15'
batch_size = 24
# Training setup
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=4e-4)

all_labels = [] # to be a list of array where each index contains labels from one CV fold
all_outputs = []
# Put ABMIL inside LOOCV loop
for i in range(k):
    print(f"K fold {i+1}/{k})")

    # Split
    train_dataset = H5Dataset(feats_path, df, "train", test_idx=i)
    test_dataset = H5Dataset(feats_path, df, "test", test_idx=i)  # single sample
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, worker_init_fn=lambda _: np.random.seed(SEED))
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    # Initialize fresh model per fold
    model = BinaryClassificationModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=4e-4)
    criterion = nn.BCEWithLogitsLoss()

    # Train model
    num_epochs = 20
    for epoch in range(num_epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = {'features': features.to(device)}, labels.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # Evaluate on held-out sample
    model.eval()
    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = {'features': features.to(device)}, labels.to(device)
            outputs = model(features)
            all_outputs.append(outputs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())


K fold 1/6)
K fold 2/6)
K fold 3/6)
K fold 4/6)
K fold 5/6)
K fold 6/6)


In [15]:
from sklearn.metrics import roc_auc_score, confusion_matrix
import numpy as np

all_labels = np.concatenate(all_labels)
all_outputs = np.concatenate(all_outputs)

# AUC
auc = roc_auc_score(all_labels, all_outputs)

# Binary predictions
preds = (all_outputs > 0).astype(int)  # logit > 0

# Accuracy
accuracy = (preds == all_labels).mean()

# Confusion matrix
cm = confusion_matrix(all_labels, preds)

print(f"LOOCV Accuracy: {accuracy:.4f}")
print(f"LOOCV AUC: {auc:.4f}")
print("Confusion Matrix:")
print(cm)

LOOCV Accuracy: 0.5172
LOOCV AUC: 0.5211
Confusion Matrix:
[[15  4]
 [10  0]]


In [16]:
np.savez("epoch_20_k_6_results", all_labels = all_labels, all_outputs = all_outputs)